# Colab GPU Worker — MoneyPrinterTurbo

Runs GPU-intensive tasks (Coqui TTS voice cloning, video rendering) for the pipeline.
Designed for zero-touch auto-launch via run_pipeline.py --auto-colab.

## Manual use
1. Runtime → Run all
2. Copy the `export COLAB_WORKER_URL=...` command to your Mac
3. Run `python run_pipeline.py --subject "..."`

In [ ]:
# ── 1. Install dependencies ──
import os, sys, json, time, requests, io, math, asyncio, uuid
from pathlib import Path

!pip install -q fastapi uvicorn pyngrok nest-asyncio TTS moviepy soundfile 2>&1 | tail -1

WORK_DIR = '/content/mpt_colab_worker'
os.makedirs(f'{WORK_DIR}/tts_cache', exist_ok=True)
os.makedirs(f'{WORK_DIR}/render_cache', exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)

os.environ['HF_HOME'] = f'{WORK_DIR}/models'
os.environ['HF_HUB_CACHE'] = f'{WORK_DIR}/models'
os.environ['TRANSFORMERS_CACHE'] = f'{WORK_DIR}/models'
print(f'Work dir: {WORK_DIR}')

In [ ]:
# ── 2. Setup ngrok tunnel ──

NGROK_AUTH_TOKEN = ""

from pyngrok import ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url

with open(f'{WORK_DIR}/url.txt', 'w') as f:
    f.write(PUBLIC_URL)

print(f'\n{"="*60}')
print(f'  COLAB WORKER URL: {PUBLIC_URL}')
print(f'{"="*60}')

In [ ]:
# ── 3. FastAPI Server (TTS + Render) ──

import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
import uvicorn

app = FastAPI(title="MPT Colab Worker")

@app.get("/health")
async def health():
    import torch
    gpu = torch.cuda.is_available()
    return {
        "status": "ok",
        "gpu": gpu,
        "gpu_name": torch.cuda.get_device_name(0) if gpu else "none",
    }

TTS_MODEL = None

@app.post("/tts")
async def do_tts(
    text: str = Form(...),
    language: str = Form("en"),
    voice_ref: UploadFile = File(...),
):
    global TTS_MODEL
    import torch, soundfile as sf

    ref_path = f"{WORK_DIR}/tts_cache/ref_{uuid.uuid4().hex[:8]}.wav"
    with open(ref_path, "wb") as f:
        f.write(await voice_ref.read())

    if TTS_MODEL is None:
        print("Loading Coqui XTTS v2 on GPU...")
        t0 = time.time()
        os.environ["COQUI_TOS_AGREED"] = "1"
        from TTS.api import TTS
        device = "cuda" if torch.cuda.is_available() else "cpu"
        TTS_MODEL = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
        print(f"Model loaded in {time.time()-t0:.1f}s on {device}")

    print(f"TTS: {len(text)} chars, lang={language}")
    t0 = time.time()
    out_path = f"{WORK_DIR}/tts_cache/out_{uuid.uuid4().hex[:8]}.wav"
    TTS_MODEL.tts_to_file(
        text=text, file_path=out_path,
        speaker_wav=ref_path, language=language, speed=1.0
    )
    elapsed = time.time() - t0
    data, sr = sf.read(out_path)
    duration = len(data) / sr
    print(f"TTS done: {duration:.0f}s audio in {elapsed:.1f}s")
    return FileResponse(out_path, media_type="audio/wav",
                       filename=f"tts_{language}.wav")

@app.post("/render")
async def do_render(
    video: UploadFile = File(...),
    audio: UploadFile = File(...),
    avatar: UploadFile = File(None),
    subtitle_entries: str = Form("[]"),
):
    import numpy as np
    from PIL import Image

    stamp = uuid.uuid4().hex[:8]
    video_path = f"{WORK_DIR}/render_cache/vid_{stamp}.mp4"
    audio_path = f"{WORK_DIR}/render_cache/aud_{stamp}.wav"
    avatar_path = None

    for p, f in [(video_path, video), (audio_path, audio)]:
        with open(p, "wb") as fh:
            fh.write(await f.read())

    if avatar and avatar.filename:
        avatar_path = f"{WORK_DIR}/render_cache/av_{stamp}.jpg"
        with open(avatar_path, "wb") as fh:
            fh.write(await avatar.read())

    subs = json.loads(subtitle_entries)

    from moviepy import CompositeVideoClip, VideoClip, VideoFileClip, AudioFileClip, TextClip

    print(f"Render: video={video.filename}, audio={audio.filename}, {len(subs)} subs")
    t0 = time.time()

    video_clip = VideoFileClip(video_path)
    audio_clip = AudioFileClip(audio_path)
    duration = audio_clip.duration
    w, h = video_clip.w, video_clip.h
    fps = video_clip.fps or 30

    overlays = []

    if avatar_path and os.path.exists(avatar_path):
        pil_img = Image.open(avatar_path).convert("RGB")
        ow = int(w * 0.28)
        oh = min(int(ow * pil_img.height / pil_img.width), int(h * 0.45))
        pil_img = pil_img.resize((ow, oh), Image.LANCZOS)
        avatar_arr = np.array(pil_img)
        mx, my = int(w * 0.03), int(h * 0.02)
        px, py = w - ow - mx, h - oh - my

        def mk_av(t):
            br = 1.0 + 0.003 * math.sin(2 * math.pi * t / 3.0)
            cw = max(1, int(ow * br))
            ch = max(1, int(oh * br))
            return np.array(Image.fromarray(avatar_arr).resize((cw, ch), Image.LANCZOS))

        overlays.append(
            VideoClip(mk_av, duration=duration)
            .with_position(lambda t: (px, py + 1.5 * math.sin(2 * math.pi * t / 2.5)))
            .with_duration(duration)
        )

    fp = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
    if not os.path.exists(fp):
        fp = "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"

    for start, end, text in subs:
        sd = end - start
        if sd <= 0: continue
        if len(text) > 60:
            words = text.split()
            lines, line = [], ""
            for w in words:
                test = f"{line} {w}".strip()
                if len(test) > 50:
                    lines.append(line); line = w
                else: line = test
            if line: lines.append(line)
            text = "\n".join(lines)
        overlays.append(
            TextClip(text=text, font_size=28, color="white", font=fp,
                     stroke_color="black", stroke_width=2, method="label")
            .with_position(("center", h * 0.85))
            .with_start(start).with_duration(sd)
        )

    final = CompositeVideoClip([video_clip] + overlays).with_audio(audio_clip)
    out_path = f"{WORK_DIR}/render_cache/final_{stamp}.mp4"
    final.write_videofile(out_path, codec="libx264", audio_codec="aac",
                          fps=fps, threads=4, preset="fast")
    for c in [video_clip, audio_clip, final]: c.close()

    elapsed = time.time() - t0
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"Render done: {size_mb:.0f}MB in {elapsed:.1f}s")
    return FileResponse(out_path, media_type="video/mp4", filename="final.mp4")


print(f"\nStarting server on port 8000...")
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")